# AI-Generated Text Detection: Fine-Tuning DeBERTa

This notebook walks through the process of fine-tuning a `microsoft/deberta-v3-small` model for detecting AI-generated text.

The methodology is based on the winning approach from the **Defactify 4.0 competition**, which involves:
1.  Using the official competition dataset from Hugging Face.
2.  Transforming the data from its original "wide" format into a "long" format suitable for binary classification.
3.  Applying a **data noising** technique to the training text to improve model robustness and generalization, as described in the winning team's paper.
4.  Fine-tuning the model using the Hugging Face `transformers` library.


In [ ]:
# This cell installs all the necessary libraries for the project.
# The 'accelerate' library helps optimize training on GPUs.
# 'fsspec' and 'huggingface_hub' are needed for loading data directly.
%pip install -q pandas torch scikit-learn transformers accelerate datasets fsspec huggingface_hub


In [ ]:
import pandas as pd
import numpy as np
import random
import string
from sklearn.model_selection import train_test_split
from datasets import load_dataset, Dataset, DatasetDict
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer

# --- Key Configuration Parameters ---
# You can change these values to experiment with different models or settings.

# Model from Hugging Face Hub
MODEL_CHECKPOINT = "microsoft/deberta-v3-small"

# Dataset from Hugging Face Hub
DATASET_NAME = "gsingh1-py/train"

# Directory to save the final fine-tuned model
OUTPUT_MODEL_DIR = "./deberta_finetuned_ai_detector"

# Percentage of words to replace with junk noise in each text sample
NOISE_LEVEL = 0.10


## Step 1: Load and Prepare the Dataset

The official competition dataset is in a "wide" format, with a separate column for the `Human_story` and each AI model's text generation.

We need to transform this into a "long" format with two columns:
1.  `text`: Contains the text sample (either human or AI).
2.  `label`: Contains the corresponding label (`0` for human, `1` for AI).

The following function handles this entire process: loading, transforming, and shuffling.


In [ ]:
def load_and_prepare_data():
    """
    Loads the wide-format data from Hugging Face, transforms it into a
    long format with text and labels, and returns it as a pandas DataFrame.
    """
    print(f"Loading dataset '{DATASET_NAME}' from Hugging Face Hub...")
    dataset = load_dataset(DATASET_NAME, split='train')
    df_wide = dataset.to_pandas()

    human_col = 'Human_story'
    ai_cols = [col for col in df_wide.columns if col not in ['prompt', human_col]]
    print(f"Found {len(ai_cols)} AI model columns to process.")

    # Create a DataFrame for human text with label 0
    df_human = df_wide[[human_col]].rename(columns={human_col: "text"})
    df_human["label"] = 0

    # "Melt" the AI text columns from wide to long format and assign label 1
    df_ai = df_wide[ai_cols].melt(var_name="source_model", value_name="text")
    df_ai["label"] = 1

    # Combine, drop any rows with missing text, and shuffle
    df_long = pd.concat([df_human, df_ai[["text", "label"]]], ignore_index=True)
    df_long.dropna(subset=["text"], inplace=True)
    df_long = df_long.sample(frac=1, random_state=42).reset_index(drop=True)

    print(f"Data prepared successfully. Total samples: {len(df_long)}")
    return df_long

# Run the function and display the first few rows of the prepared data
prepared_df = load_and_prepare_data()
prepared_df.head()


## Step 2: Apply Data Noising

To make our model more robust, we will deliberately inject random "junk words" into the training text. This technique helps the model generalize better and not overfit to specific phrasing. It forces the model to learn the deeper semantic patterns of AI vs. human text rather than superficial artifacts.


In [ ]:
def add_noise(text, noise_level=0.1):
    """Injects random junk words into a text."""
    words = str(text).split()
    if not words:
        return ""
    num_noise_words = int(len(words) * noise_level)
    for _ in range(num_noise_words):
        random_index = random.randint(0, len(words))
        junk_word = ''.join(random.choice(string.ascii_lowercase) for _ in range(random.randint(3, 8)))
        words.insert(random_index, junk_word)
    return " ".join(words)

# Apply the noise function to our prepared dataframe
print(f"Applying noise to the dataset at a {NOISE_LEVEL*100}% level...")
prepared_df["text"] = prepared_df["text"].apply(lambda x: add_noise(x, noise_level=NOISE_LEVEL))

print("Noise application complete. Here's a sample of the noised text:")
prepared_df.head()


## Step 3: Tokenize the Dataset

Transformer models like DeBERTa don't work with raw text. They require the text to be converted into numerical tokens. We will use the model's specific tokenizer to handle this process.


In [ ]:
# Convert the pandas DataFrame back into a Hugging Face Dataset
dataset = Dataset.from_pandas(prepared_df)

# Split the dataset into training and testing sets
train_test_split = dataset.train_test_split(test_size=0.2, seed=42, stratify_by_column="label")
dataset_dict = DatasetDict({
    'train': train_test_split['train'],
    'test': train_test_split['test']
})

# Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)

def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=512)

# Apply the tokenizer to the entire dataset
tokenized_datasets = dataset_dict.map(tokenize_function, batched=True)
print("Tokenization complete.")


## Step 4: Fine-Tune the DeBERTa Model

Now we will set up and run the fine-tuning process using the Hugging Face `Trainer`.

- **`TrainingArguments`**: This class holds all the hyperparameters for training, such as learning rate, batch size, and number of epochs.
- **`Trainer`**: This class handles the entire training and evaluation loop.


In [ ]:
# Load the pre-trained model
model = AutoModelForSequenceClassification.from_pretrained(MODEL_CHECKPOINT, num_labels=2)

# Define the training arguments
training_args = TrainingArguments(
    output_dir=OUTPUT_MODEL_DIR,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    logging_dir='./logs',
    logging_steps=100,
)

# Create the Trainer instance
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
)

# Start the training
print("--- Starting model fine-tuning ---")
trainer.train()
print("--- Training complete ---")

# Save the best performing model to the output directory
trainer.save_model(OUTPUT_MODEL_DIR)
tokenizer.save_pretrained(OUTPUT_MODEL_DIR)
print(f"Best model saved to {OUTPUT_MODEL_DIR}")


## Conclusion and Next Steps

The fine-tuning process is complete. The best version of the model, along with its tokenizer, has been saved to the `./deberta_finetuned_ai_detector` directory.

**Next Steps:**
- This model can now be loaded from the saved directory for inference on new, unseen text.
- For a full research paper, you would perform a final evaluation on the official hold-out test set.
- You could also apply **post-hoc calibration** to the model's outputs to ensure its predicted probabilities are reliable.
